# Standalone-comment benchmark

Inspect stored held-out predictions for the five text configurations and reproduce seed-42 Macro F1. This subset was filtered for annotation agreement.

**Scope:** These notebooks recompute descriptive statistics from the archived data and predictions. They do not retrain the original encoders or produce new evaluation populations.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score, average_precision_score, brier_score_loss

ROOT = Path.cwd().resolve()
if not (ROOT / "CITATION.cff").exists():
    ROOT = ROOT.parent
assert (ROOT / "CITATION.cff").exists(), "Run from the repository root or notebooks/"
print("Project:", ROOT.name)

## Published model metrics

In [ ]:
m = pd.read_csv(ROOT / "results/comments/benchmark_v1.1/TEXT_TEST_METRICS.csv")
view = m[m.seed == 42][["model","n","accuracy","macro_f1","roc_auc","average_precision"]].sort_values("macro_f1",ascending=False)
display(view.reset_index(drop=True))

## Independent Macro F1 from saved predictions

In [ ]:
p = pd.read_csv(ROOT / "results/comments/benchmark_v1.1/TEXT_TEST_PREDICTIONS.csv")
p42 = p[p.seed == 42]
rows = []
for name,sub in p42.groupby("model"):
    rows.append({"model":name,"n":len(sub),"macro_f1_recomputed":f1_score(sub.label_final,sub.prediction,average="macro",labels=[0,1]),"accuracy_recomputed":(sub.label_final==sub.prediction).mean()})
recomputed=pd.DataFrame(rows).sort_values("macro_f1_recomputed",ascending=False)
display(recomputed)
assert (recomputed.n==1226).all()
assert abs(recomputed.loc[recomputed.model.eq("equal_text_ensemble"),"macro_f1_recomputed"].iat[0]-0.966271)<1e-5

## Descriptive plot and recorded inference
The grouped tests are archived, and confidence intervals are not recomputed here.

In [ ]:
fig,ax=plt.subplots(figsize=(8,3.6))
ax.barh(view.model.iloc[::-1],view.macro_f1.iloc[::-1])
ax.set_xlabel("Macro F1 (held-out comments)")
ax.set_xlim(0,1.0)
ax.set_title("Standalone-comment benchmark, seed 42")
plt.tight_layout();plt.show()
display(pd.read_csv(ROOT / "results/comments/benchmark_v1.1/TEXT_PAIRED_GROUP_STATISTICS.csv").head())